In [1]:
!nvidia-smi | head -5
import sys, torch
print("Python:", sys.version.split()[0])
print("torch:", torch.__version__, "cuda:", torch.version.cuda)

Thu Apr 23 23:52:56 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
Python: 3.12.13
torch: 2.10.0+cu128 cuda: 12.8


In [2]:
!pip install -q torch_geometric
!pip install -q pyg_lib torch_scatter torch_sparse torch_cluster \
    -f https://data.pyg.org/whl/torch-2.10.0+cu128.html

# 验证
import torch_geometric, torch_scatter
from torch_scatter import scatter_add
print("pyg:", torch_geometric.__version__)
x = torch.randn(5, 3).cuda(); idx = torch.tensor([0,0,1,1,2]).cuda()
print("scatter OK:", scatter_add(x, idx, dim=0).shape)

pyg: 2.7.0
scatter OK: torch.Size([3, 3])


In [3]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/lrgb_mini/data', exist_ok=True)
DATA_ROOT = '/content/drive/MyDrive/lrgb_mini/data'  # 数据集缓存到 Drive，重连不丢

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.datasets import LRGBDataset
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, GPSConv, GINEConv, global_mean_pool
from torch_geometric.transforms import AddRandomWalkPE

# ---- 先清掉之前半处理的缓存（之前用 LapPE 写进去的，不能混用）----
import shutil, os
proc_dir = f"{DATA_ROOT}/peptides-func/processed"
if os.path.exists(proc_dir):
    shutil.rmtree(proc_dir)
    print(f"cleared old processed cache: {proc_dir}")

# ---- 用 RWSE (Random Walk SE) 代替 LapPE ----
# 优点：对任何 graph 大小都 work，Tönshoff 配置也用这个
PE_DIM = 20
pe_transform = AddRandomWalkPE(walk_length=PE_DIM, attr_name='pe')
print(f"Using RWSE with walk_length={PE_DIM}")

# ---- Dataset ----
name = 'Peptides-func'
train_ds = LRGBDataset(DATA_ROOT, name=name, split='train', pre_transform=pe_transform)
val_ds   = LRGBDataset(DATA_ROOT, name=name, split='val',   pre_transform=pe_transform)
test_ds  = LRGBDataset(DATA_ROOT, name=name, split='test',  pre_transform=pe_transform)

print(f"train/val/test: {len(train_ds)}/{len(val_ds)}/{len(test_ds)}")
print(f"node feat dim: {train_ds.num_node_features}")
print(f"edge feat dim: {train_ds.num_edge_features}")
print(f"num classes: {train_ds.num_classes}")

# ---- 组件 ----
class AtomEncoder(nn.Module):
    def __init__(self, in_dim, hidden_dim, pe_dim=PE_DIM):
        super().__init__()
        self.lin = nn.Linear(in_dim, hidden_dim - pe_dim)
        self.pe_lin = nn.Linear(pe_dim, pe_dim)
    def forward(self, x, pe):
        return torch.cat([self.lin(x), self.pe_lin(pe)], dim=-1)

class BondEncoder(nn.Module):
    def __init__(self, in_dim, hidden_dim):
        super().__init__()
        self.lin = nn.Linear(in_dim, hidden_dim)
    def forward(self, e):
        return self.lin(e)

class MLPHead(nn.Module):
    """Tönshoff 的关键改动：2 层 MLP head"""
    def __init__(self, hidden_dim, out_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, out_dim),
        )
    def forward(self, x): return self.net(x)

# ---- Model 1: GCN (traditional MPGNN) ----
class GCNModel(nn.Module):
    def __init__(self, node_dim, edge_dim, num_classes,
                 hidden=240, num_layers=6, dropout=0.1):
        super().__init__()
        # hidden 改成 240 让它能被 PE_DIM(20) 减下来还是偶数
        self.atom_enc = AtomEncoder(node_dim, hidden)
        self.convs = nn.ModuleList([GCNConv(hidden, hidden) for _ in range(num_layers)])
        self.norms = nn.ModuleList([nn.BatchNorm1d(hidden) for _ in range(num_layers)])
        self.dropout = dropout
        self.head = MLPHead(hidden, num_classes)
    def forward(self, data):
        x = self.atom_enc(data.x.float(), data.pe.float())
        ei = data.edge_index
        for conv, norm in zip(self.convs, self.norms):
            h = conv(x, ei)
            h = norm(h)
            h = F.relu(h)
            h = F.dropout(h, p=self.dropout, training=self.training)
            x = x + h   # skip around non-linearity (Tönshoff fix)
        g = global_mean_pool(x, data.batch)
        return self.head(g)

# ---- Model 2: GPS (hybrid) ----
class GPSModel(nn.Module):
    def __init__(self, node_dim, edge_dim, num_classes,
                 hidden=96, num_layers=4, heads=4, dropout=0.1):
        super().__init__()
        self.atom_enc = AtomEncoder(node_dim, hidden)
        self.bond_enc = BondEncoder(edge_dim, hidden)
        self.convs = nn.ModuleList()
        for _ in range(num_layers):
            mpnn = GINEConv(nn.Sequential(
                nn.Linear(hidden, hidden), nn.ReLU(), nn.Linear(hidden, hidden)
            ))
            self.convs.append(GPSConv(hidden, conv=mpnn, heads=heads, dropout=dropout))
        self.head = MLPHead(hidden, num_classes)
    def forward(self, data):
        x = self.atom_enc(data.x.float(), data.pe.float())
        e = self.bond_enc(data.edge_attr.float())
        for conv in self.convs:
            x = conv(x, data.edge_index, data.batch, edge_attr=e)
        g = global_mean_pool(x, data.batch)
        return self.head(g)

print("Models defined.")

cleared old processed cache: /content/drive/MyDrive/lrgb_mini/data/peptides-func/processed
Using RWSE with walk_length=20


Processing...
Processing test dataset: 100%|██████████| 2331/2331 [00:03<00:00, 630.11it/s]
Done!


train/val/test: 10873/2331/2331
node feat dim: 9
edge feat dim: 3
num classes: 10
Models defined.


In [6]:
from torchmetrics.classification import MultilabelAveragePrecision
import time

device = 'cuda'

def train_model(model, train_loader, val_loader, test_loader,
                lr=1e-3, wd=0.0, epochs=200, patience=40):
    model = model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    loss_fn = nn.BCEWithLogitsLoss()
    ap_metric = MultilabelAveragePrecision(num_labels=train_ds.num_classes,
                                           average='macro').to(device)

    best_val_ap, best_test_ap, bad = 0.0, 0.0, 0
    for epoch in range(1, epochs + 1):
        # ---- train ----
        model.train(); t0 = time.time()
        total_loss = 0
        for data in train_loader:
            data = data.to(device)
            opt.zero_grad()
            out = model(data)
            loss = loss_fn(out, data.y.float())
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            total_loss += loss.item() * data.num_graphs
        sched.step()
        train_loss = total_loss / len(train_loader.dataset)

        # ---- eval helper ----
        def eval_loader(loader):
            model.eval(); ap_metric.reset()
            with torch.no_grad():
                for data in loader:
                    data = data.to(device)
                    out = model(data)
                    ap_metric.update(out, data.y.int())
            return ap_metric.compute().item()

        val_ap = eval_loader(val_loader)
        if val_ap > best_val_ap:
            best_val_ap = val_ap
            best_test_ap = eval_loader(test_loader)
            bad = 0
        else:
            bad += 1

        if epoch % 10 == 0 or epoch == 1:
            print(f"ep {epoch:3d} | loss {train_loss:.4f} | "
                  f"val AP {val_ap:.4f} | best val {best_val_ap:.4f} "
                  f"| test@best {best_test_ap:.4f} | {time.time()-t0:.1f}s")

        if bad >= patience:
            print(f"early stop at epoch {epoch}")
            break

    return best_val_ap, best_test_ap

In [7]:
train_loader = DataLoader(train_ds, batch_size=128, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=128, num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=128, num_workers=2)

NODE_DIM = train_ds.num_node_features
EDGE_DIM = train_ds.num_edge_features
NUM_CLS  = train_ds.num_classes

results = {}

torch.manual_seed(42)
print("=" * 60); print("Training GCN (traditional MPGNN)"); print("=" * 60)
gcn = GCNModel(NODE_DIM, EDGE_DIM, NUM_CLS)
print(f"params: {sum(p.numel() for p in gcn.parameters()):,}")
results['GCN'] = train_model(gcn, train_loader, val_loader, test_loader)

torch.manual_seed(42)
print("\n" + "=" * 60); print("Training GPS (hybrid)"); print("=" * 60)
gps = GPSModel(NODE_DIM, EDGE_DIM, NUM_CLS)
print(f"params: {sum(p.numel() for p in gps.parameters()):,}")
results['GPS'] = train_model(gps, train_loader, val_loader, test_loader)

print("\n\n============ FINAL RESULTS ============")
print(f"{'Model':<8} {'Val AP':<10} {'Test AP':<10}")
for m, (v, t) in results.items():
    print(f"{m:<8} {v:.4f}     {t:.4f}")

Training GCN (traditional MPGNN)
params: 412,790
ep   1 | loss 0.3225 | val AP 0.3319 | best val 0.3319 | test@best 0.3282 | 4.5s
ep  10 | loss 0.2544 | val AP 0.3905 | best val 0.4060 | test@best 0.3977 | 3.2s
ep  20 | loss 0.2393 | val AP 0.4566 | best val 0.5187 | test@best 0.4888 | 2.7s
ep  30 | loss 0.2275 | val AP 0.5010 | best val 0.5254 | test@best 0.5095 | 3.2s
ep  40 | loss 0.2179 | val AP 0.5378 | best val 0.5510 | test@best 0.5177 | 3.1s
ep  50 | loss 0.2051 | val AP 0.4918 | best val 0.5710 | test@best 0.5416 | 2.8s
ep  60 | loss 0.1946 | val AP 0.5433 | best val 0.5835 | test@best 0.5526 | 3.2s
ep  70 | loss 0.1879 | val AP 0.4808 | best val 0.6069 | test@best 0.5640 | 3.3s
ep  80 | loss 0.1811 | val AP 0.5624 | best val 0.6069 | test@best 0.5640 | 3.2s
ep  90 | loss 0.1750 | val AP 0.6211 | best val 0.6240 | test@best 0.5871 | 3.2s
ep 100 | loss 0.1687 | val AP 0.6122 | best val 0.6240 | test@best 0.5871 | 3.1s
ep 110 | loss 0.1644 | val AP 0.6248 | best val 0.6331 | tes